# Tech Challenge — Fase 3
## Ajustes metodológicos finais

Esta versão fecha os pontos de aderência mais importantes da entrega:

- continuidade explícita com a Fase 2;
- features escolares/contextuais sem usar proficiência ou target;
- baseline ingênuo formal;
- Balanced Accuracy e PR-AUC;
- análise prospectiva de metas 2030;
- geração automática de artefatos e figuras.

### Sobre a granularidade

O identificador `id_escola` da avaliação de alfabetização não apresenta correspondência
com o código de escola do Censo Escolar no cruzamento testado. Por isso, a versão final
não força uma integração inválida.

Em vez disso, cria **features operacionais da própria escola**, derivadas somente de
campos administrativos não relacionados ao resultado (`presenca`, `preenchimento_caderno`,
quantidade de registros e cadernos). Essas features variam por escola e quebram a
replicação puramente município/rede.

A predição continua sendo contextual: alunos da mesma escola podem compartilhar o mesmo
vetor escolar. Portanto, o modelo é apresentado como **classificação de risco em nível
de registro de aluno com contexto escolar e territorial**, e não como diagnóstico
pedagógico individual.


In [ ]:
!pip install -q google-cloud-bigquery db-dtypes pyarrow scikit-learn shap joblib pandas numpy matplotlib

from pathlib import Path
import json
import hashlib
import os
import time
import warnings
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import randint, loguniform
from google.colab import auth
from google.cloud import bigquery

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)
import shap
import joblib

warnings.filterwarnings("ignore")

PROJECT_ID = (
    os.getenv("GCP_PROJECT_ID")
    or os.getenv("GOOGLE_CLOUD_PROJECT")
    or "fiap-techchallenge-fase2"
)

# Reprodutibilidade:
# - na execução de referência foi usado fiap-techchallenge-fase2;
# - outro usuário pode definir GCP_PROJECT_ID para um projeto em que possua
#   permissão bigquery.jobs.create, sem alterar as fontes públicas consultadas.
PHASE2_REPO = "https://github.com/acorrea79/techchallenge-fase2-pipeline-alfabetizacao"

OUT = Path("/content/tech_challenge_fase3_v3")
REPORTS = OUT / "reports"
IMAGES = OUT / "images"
DATA = OUT / "data"
MODELS = OUT / "models"

for p in [REPORTS, IMAGES, DATA, MODELS]:
    p.mkdir(parents=True, exist_ok=True)

auth.authenticate_user()
client = bigquery.Client(project=PROJECT_ID)
print("Projeto usado para executar jobs BigQuery:", PROJECT_ID)
print("Para usar outro projeto: defina GCP_PROJECT_ID antes desta célula.")


## 1. Proveniência da Fase 2

A Fase 2 é a origem da arquitetura analítica e das regras de negócio territoriais.

As Golds municipais e de metas da Fase 2 não são usadas diretamente como features
do classificador porque contêm agregações/resultados contemporâneos, como taxa de
alfabetização, distância à meta, status de meta e score de prioridade.

A Fase 3 preserva a linhagem da mesma avaliação, constrói a Gold individual necessária
ao target solicitado e reutiliza as metas/indicadores na camada de inteligência territorial.


In [ ]:
lineage = pd.DataFrame([
    ["gold_base_ia_alfabetizacao.parquet",
     "referência de linhagem; agregada em nível territorial",
     "não substitui a Gold individual exigida pelo target da Fase 3"],
    ["gold_comparativo_meta_resultado_municipio.parquet",
     "referência da camada de metas e política pública",
     "não entra no classificador para evitar leakage"],
    ["gold_ranking_municipios_prioritarios / indicadores",
     "referência para priorização territorial",
     "estendido na Fase 3 para análise prospectiva 2030"],
], columns=["artefato_fase2","papel_na_fase3","justificativa"])

display(lineage)
lineage.to_csv(REPORTS / "fase2_lineage.csv", index=False)


## 2. Auditoria do identificador escolar

O cruzamento anterior com o Censo Escolar obteve cobertura zero. Esta célula documenta
o motivo operacional e evita repetir um join sem correspondência.


In [ ]:
audit_school_sql = '''
WITH a AS (
  SELECT DISTINCT
    CAST(id_escola AS STRING) AS id_escola_avaliacao
  FROM `basedosdados.br_inep_avaliacao_alfabetizacao.alunos`
  WHERE id_escola IS NOT NULL
  LIMIT 20000
),
c AS (
  SELECT DISTINCT
    CAST(id_escola AS STRING) AS id_escola_censo
  FROM `basedosdados.br_inep_censo_escolar.escola`
  WHERE ano = 2022
)
SELECT
  COUNT(*) AS ids_avaliacao_amostrados,
  COUNTIF(c.id_escola_censo IS NOT NULL) AS ids_com_match_censo
FROM a
LEFT JOIN c
  ON a.id_escola_avaliacao = c.id_escola_censo
'''

audit_school = client.query(audit_school_sql).to_dataframe(create_bqstorage_client=False)
display(audit_school)

school_id_match_rate = (
    100 * audit_school.loc[0, "ids_com_match_censo"]
    / audit_school.loc[0, "ids_avaliacao_amostrados"]
)

print("SCHOOL_ID_MATCH_RATE_PCT=", round(school_id_match_rate, 4))
print("Decisão: não usar Censo Escolar via id_escola quando não houver correspondência de chave.")


## 3. Gold ML individual com contexto escolar operacional

As features escolares são calculadas diretamente da mesma fonte da avaliação, mas **sem**
usar `alfabetizado` ou `proficiencia`.

São consideradas disponíveis após a aplicação/participação e antes do uso do resultado
de alfabetização:

- tamanho da escola na avaliação;
- taxa de presença;
- taxa de preenchimento do caderno;
- taxa de elegibilidade para análise;
- quantidade de cadernos;
- quantidade de escolas participantes no município/rede;
- participação da escola no total de registros do município/rede.

Isso introduz variação real entre escolas dentro de um mesmo município/rede.


In [ ]:
gold_sql = '''
WITH source AS (
  SELECT
    CAST(ano AS INT64) AS ano,
    CAST(id_municipio AS STRING) AS id_municipio,
    CAST(id_escola AS STRING) AS id_escola,
    CAST(id_aluno AS STRING) AS id_aluno,
    CAST(rede AS STRING) AS rede,
    CAST(caderno AS STRING) AS caderno,
    CAST(presenca AS STRING) AS presenca,
    CAST(preenchimento_caderno AS STRING) AS preenchimento_caderno,
    CASE
      WHEN CAST(alfabetizado AS STRING) = '0' THEN 1
      WHEN CAST(alfabetizado AS STRING) = '1' THEN 0
      ELSE NULL
    END AS target_nao_alfabetizado
  FROM `basedosdados.br_inep_avaliacao_alfabetizacao.alunos`
),
school_context AS (
  SELECT
    ano,
    id_escola,
    COUNT(*) AS escola_total_registros,
    AVG(CASE WHEN presenca='1' THEN 1.0 ELSE 0.0 END) AS escola_taxa_presenca,
    AVG(CASE WHEN preenchimento_caderno='1' THEN 1.0 ELSE 0.0 END) AS escola_taxa_preenchimento,
    AVG(CASE WHEN presenca='1' AND preenchimento_caderno='1' THEN 1.0 ELSE 0.0 END) AS escola_taxa_elegiveis,
    COUNT(DISTINCT caderno) AS escola_qtd_cadernos
  FROM source
  WHERE id_escola IS NOT NULL
  GROUP BY 1,2
),
municip_context AS (
  SELECT
    ano,
    id_municipio,
    rede,
    COUNT(*) AS municipio_total_registros,
    COUNT(DISTINCT id_escola) AS municipio_qtd_escolas_avaliacao
  FROM source
  WHERE id_municipio IS NOT NULL
  GROUP BY 1,2,3
),
base AS (
  SELECT
    ano, id_municipio, id_escola, id_aluno, rede, target_nao_alfabetizado
  FROM source
  WHERE presenca='1'
    AND preenchimento_caderno='1'
    AND target_nao_alfabetizado IS NOT NULL
    AND MOD(FARM_FINGERPRINT(CONCAT(
      CAST(ano AS STRING), '|',
      id_municipio, '|',
      id_escola, '|',
      id_aluno
    )), 10) = 0
),
socio AS (
  SELECT
    CAST(pib.id_municipio AS STRING) AS id_municipio,
    CAST(pib.ano AS INT64) AS ano_socio,
    CAST(pop.populacao AS FLOAT64) AS populacao,
    SAFE_DIVIDE(
      CAST(pib.pib AS FLOAT64),
      NULLIF(CAST(pop.populacao AS FLOAT64), 0)
    ) AS pib_per_capita
  FROM `basedosdados.br_ibge_pib.municipio` pib
  INNER JOIN `basedosdados.br_ibge_populacao.municipio` pop
    ON CAST(pib.id_municipio AS STRING)=CAST(pop.id_municipio AS STRING)
   AND CAST(pib.ano AS INT64)=CAST(pop.ano AS INT64)
),
enriched AS (
  SELECT
    b.*,
    s.ano_socio,
    s.populacao,
    s.pib_per_capita,
    sc.escola_total_registros,
    sc.escola_taxa_presenca,
    sc.escola_taxa_preenchimento,
    sc.escola_taxa_elegiveis,
    sc.escola_qtd_cadernos,
    mc.municipio_total_registros,
    mc.municipio_qtd_escolas_avaliacao,
    SAFE_DIVIDE(sc.escola_total_registros, mc.municipio_total_registros)
      AS escola_participacao_municipio,
    ROW_NUMBER() OVER (
      PARTITION BY b.ano,b.id_municipio,b.id_escola,b.id_aluno
      ORDER BY s.ano_socio DESC
    ) AS rn
  FROM base b
  LEFT JOIN school_context sc
    ON b.ano=sc.ano AND b.id_escola=sc.id_escola
  LEFT JOIN municip_context mc
    ON b.ano=mc.ano
   AND b.id_municipio=mc.id_municipio
   AND b.rede=mc.rede
  LEFT JOIN socio s
    ON s.id_municipio=b.id_municipio
   AND s.ano_socio<b.ano
)
SELECT * EXCEPT(rn)
FROM enriched
WHERE rn=1
ORDER BY ano,id_municipio,id_escola,id_aluno
'''

gold = client.query(gold_sql).to_dataframe(create_bqstorage_client=False)

UF_CODE_TO_SIGLA = {
    '11':'RO','12':'AC','13':'AM','14':'RR','15':'PA','16':'AP','17':'TO',
    '21':'MA','22':'PI','23':'CE','24':'RN','25':'PB','26':'PE','27':'AL','28':'SE','29':'BA',
    '31':'MG','32':'ES','33':'RJ','35':'SP','41':'PR','42':'SC','43':'RS',
    '50':'MS','51':'MT','52':'GO','53':'DF'
}
UF_TO_REGIAO = {
    **{u:'Norte' for u in ['RO','AC','AM','RR','PA','AP','TO']},
    **{u:'Nordeste' for u in ['MA','PI','CE','RN','PB','PE','AL','SE','BA']},
    **{u:'Sudeste' for u in ['MG','ES','RJ','SP']},
    **{u:'Sul' for u in ['PR','SC','RS']},
    **{u:'Centro-Oeste' for u in ['MS','MT','GO','DF']},
}

gold["id_municipio"] = gold["id_municipio"].astype("string").str.zfill(7)
gold["id_escola"] = gold["id_escola"].astype("string")
gold["id_aluno"] = gold["id_aluno"].astype("string")
gold["rede"] = gold["rede"].astype("string")
gold["sigla_uf"] = gold["id_municipio"].str[:2].map(UF_CODE_TO_SIGLA).astype("string")
gold["regiao"] = gold["sigla_uf"].map(UF_TO_REGIAO).astype("string")

KEY = ["ano","id_municipio","id_escola","id_aluno"]
gold = gold.sort_values(KEY, kind="mergesort").reset_index(drop=True)

hash_bytes = pd.util.hash_pandas_object(gold, index=False).to_numpy(dtype="uint64").tobytes()
gold_sha256 = hashlib.sha256(hash_bytes).hexdigest().upper()

gold.to_parquet(DATA / "gold_ml_final.parquet", index=False)

print("GOLD_SHAPE=", gold.shape)
print("GOLD_SHA256=", gold_sha256)
print("NULOS_TOTAL=", int(gold.isna().sum().sum()))
print("DUPLICADOS=", int(gold.duplicated(KEY).sum()))


## 4. Validação da nova granularidade


In [ ]:
school_feature_cols = [
    "escola_total_registros",
    "escola_taxa_presenca",
    "escola_taxa_preenchimento",
    "escola_taxa_elegiveis",
    "escola_qtd_cadernos",
    "municipio_qtd_escolas_avaliacao",
    "escola_participacao_municipio",
]

school_feature_coverage = 100 * gold[school_feature_cols].notna().all(axis=1).mean()

# Dentro de cada município/rede, medir se há pelo menos dois vetores escolares distintos.
school_vectors = (
    gold[
        ["ano","id_municipio","rede","id_escola"] + school_feature_cols
    ]
    .drop_duplicates(["ano","id_municipio","rede","id_escola"])
)

vector_counts = (
    school_vectors.groupby(["ano","id_municipio","rede"])[school_feature_cols]
    .nunique(dropna=False)
    .max(axis=1)
)

school_variation_pct = 100 * (vector_counts > 1).mean()

print("SCHOOL_FEATURE_COVERAGE_PCT=", round(school_feature_coverage, 2))
print("MUNICIPIO_REDE_WITH_SCHOOL_VARIATION_PCT=", round(school_variation_pct, 2))

granularity = pd.DataFrame({
    "indicador": [
        "linhas_gold",
        "cobertura_features_escola_pct",
        "grupos_municipio_rede_com_variacao_escolar_pct",
        "escolas_distintas_2023",
        "escolas_distintas_2024",
    ],
    "valor": [
        len(gold),
        school_feature_coverage,
        school_variation_pct,
        gold.loc[gold.ano==2023,"id_escola"].nunique(),
        gold.loc[gold.ano==2024,"id_escola"].nunique(),
    ]
})
display(granularity.round(2))
granularity.to_csv(REPORTS / "granularity_audit.csv", index=False)


## 5. Análise Exploratória de Dados (EDA)

A modelagem é precedida por uma EDA objetiva para verificar distribuição do target,
diferenças territoriais, comportamento das variáveis numéricas, contexto escolar e
associações úteis para formular hipóteses.

As análises desta seção são descritivas. Nenhuma conclusão é tratada como causal.


In [ ]:
# 5.1 Distribuição do target por ano
target_by_year = (
    gold.groupby(["ano", "target_nao_alfabetizado"])
        .size().rename("total").reset_index()
)
target_by_year["percentual"] = (
    target_by_year.groupby("ano")["total"]
    .transform(lambda s: 100 * s / s.sum())
)
display(target_by_year.round(2))
target_by_year.to_csv(REPORTS / "eda_target_by_year.csv", index=False)

pivot_target = target_by_year.pivot(
    index="ano", columns="target_nao_alfabetizado", values="percentual"
).fillna(0)

fig, ax = plt.subplots(figsize=(7.2, 4.5))
x = np.arange(len(pivot_target.index))
width = 0.35
ax.bar(x - width/2, pivot_target.get(0, 0), width, label="Alfabetizado")
ax.bar(x + width/2, pivot_target.get(1, 0), width, label="Não alfabetizado")
ax.set_xticks(x, [str(v) for v in pivot_target.index])
ax.set_ylabel("% de registros")
ax.set_title("Distribuição do target por ano")
ax.legend()
fig.tight_layout()
fig.savefig(IMAGES / "eda_target_by_year.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
# 5.2 Prevalência por rede, região e UF
rede_map = {"2": "Estadual", "3": "Municipal", "4": "Privada"}

eda_rede = (
    gold.assign(rede_nome=gold["rede"].map(rede_map).fillna(gold["rede"]))
        .groupby(["ano", "rede_nome"])["target_nao_alfabetizado"]
        .agg(["count", "mean"]).reset_index()
        .rename(columns={"count": "registros", "mean": "taxa_risco"})
)
eda_rede["taxa_risco_pct"] = 100 * eda_rede["taxa_risco"]

eda_regiao = (
    gold.groupby(["ano", "regiao"])["target_nao_alfabetizado"]
        .agg(["count", "mean"]).reset_index()
        .rename(columns={"count": "registros", "mean": "taxa_risco"})
)
eda_regiao["taxa_risco_pct"] = 100 * eda_regiao["taxa_risco"]

eda_uf_2024 = (
    gold[gold["ano"] == 2024]
        .groupby("sigla_uf")["target_nao_alfabetizado"]
        .agg(["count", "mean"]).reset_index()
        .rename(columns={"count": "registros", "mean": "taxa_risco"})
)
eda_uf_2024["taxa_risco_pct"] = 100 * eda_uf_2024["taxa_risco"]
eda_uf_2024 = eda_uf_2024.sort_values("taxa_risco_pct", ascending=False)

print("Risco por rede:")
display(eda_rede.round(2))
print("Risco por região:")
display(eda_regiao.round(2))
print("UFs — 2024:")
display(eda_uf_2024.head(10).round(2))

eda_rede.to_csv(REPORTS / "eda_risk_by_network.csv", index=False)
eda_regiao.to_csv(REPORTS / "eda_risk_by_region.csv", index=False)
eda_uf_2024.to_csv(REPORTS / "eda_risk_by_uf_2024.csv", index=False)

fig, ax = plt.subplots(figsize=(8.4, 4.8))
reg_pivot = eda_regiao.pivot(index="regiao", columns="ano", values="taxa_risco_pct")
reg_pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("% não alfabetizado")
ax.set_title("Taxa de não alfabetização por região")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(IMAGES / "eda_risk_by_region.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
# 5.3 Distribuições numéricas e outliers
eda_numeric = [
    "populacao", "pib_per_capita", "escola_total_registros",
    "escola_taxa_presenca", "escola_taxa_preenchimento",
    "escola_qtd_cadernos", "municipio_qtd_escolas_avaliacao",
    "escola_participacao_municipio",
]

numeric_summary = gold[eda_numeric].describe(
    percentiles=[0.01,0.05,0.25,0.50,0.75,0.95,0.99]
).T
display(numeric_summary.round(3))
numeric_summary.to_csv(REPORTS / "eda_numeric_summary.csv")

plot_df = gold.sample(n=min(50000, len(gold)), random_state=42).copy()
plot_df["log_populacao"] = np.log1p(plot_df["populacao"])
plot_df["log_pib_per_capita"] = np.log1p(plot_df["pib_per_capita"])

fig, axes = plt.subplots(1,2,figsize=(10.5,4.4))
axes[0].hist(plot_df["log_populacao"].dropna(), bins=40)
axes[0].set_title("População municipal — log1p")
axes[0].set_xlabel("log1p(população)")
axes[1].hist(plot_df["log_pib_per_capita"].dropna(), bins=40)
axes[1].set_title("PIB per capita — log1p")
axes[1].set_xlabel("log1p(PIB per capita)")
fig.tight_layout()
fig.savefig(IMAGES / "eda_numeric_distributions.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
# 5.4 Correlações lineares exploratórias
corr_cols = [
    "target_nao_alfabetizado", "populacao", "pib_per_capita",
    "escola_total_registros", "escola_taxa_presenca",
    "escola_taxa_preenchimento", "escola_taxa_elegiveis",
    "escola_qtd_cadernos", "municipio_qtd_escolas_avaliacao",
    "escola_participacao_municipio",
]
corr = gold[corr_cols].corr(numeric_only=True)
target_corr = (
    corr["target_nao_alfabetizado"]
    .drop("target_nao_alfabetizado")
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .rename("correlacao_com_target").reset_index()
    .rename(columns={"index":"variavel"})
)
display(target_corr.round(4))
target_corr.to_csv(REPORTS / "eda_correlations_with_target.csv", index=False)

fig, ax = plt.subplots(figsize=(8.5,4.8))
plot_corr = target_corr.sort_values("correlacao_com_target")
ax.barh(plot_corr["variavel"], plot_corr["correlacao_com_target"])
ax.axvline(0, linewidth=0.8)
ax.set_xlabel("Correlação de Pearson com o target")
ax.set_title("Associações lineares exploratórias")
fig.tight_layout()
fig.savefig(IMAGES / "eda_correlations.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
# 5.5 Contexto escolar por classe
school_context_by_class = (
    gold.groupby(["ano","target_nao_alfabetizado"])[[
        "escola_total_registros", "escola_taxa_presenca",
        "escola_taxa_preenchimento", "escola_qtd_cadernos",
        "escola_participacao_municipio",
    ]].median().reset_index()
)
display(school_context_by_class.round(4))
school_context_by_class.to_csv(
    REPORTS / "eda_school_context_by_class.csv", index=False
)


In [ ]:
# 5.6 Hipóteses que orientam a modelagem
hypotheses = pd.DataFrame([
    {
        "hipotese":"H1",
        "descricao":"O risco apresenta forte heterogeneidade territorial.",
        "decisao_modelagem":"manter UF como feature e região para EDA/negócio"
    },
    {
        "hipotese":"H2",
        "descricao":"O contexto operacional da escola adiciona variação além de município/rede.",
        "decisao_modelagem":"usar presença, preenchimento, volume, cadernos e participação escolar"
    },
    {
        "hipotese":"H3",
        "descricao":"População e PIB per capita têm distribuição assimétrica e efeito possivelmente não linear.",
        "decisao_modelagem":"comparar modelos lineares e não lineares dentro do mesmo pipeline"
    },
])
display(hypotheses)
hypotheses.to_csv(REPORTS / "eda_hypotheses.csv", index=False)


## 6. Baseline e comparação de modelos

O baseline `Dummy_All_Risk` prevê todos os alunos como não alfabetizados.

A métrica primária para seleção é **PR-AUC**, porque mede a capacidade de ordenar
casos de risco sem depender do threshold. F1, Balanced Accuracy, Recall e ROC-AUC
também são reportados.


In [ ]:
TARGET = "target_nao_alfabetizado"

CATEGORICAL_FEATURES = ["rede","sigla_uf"]
NUMERIC_FEATURES = [
    "populacao",
    "pib_per_capita",
    "escola_total_registros",
    "escola_taxa_presenca",
    "escola_taxa_preenchimento",
    "escola_taxa_elegiveis",
    "escola_qtd_cadernos",
    "municipio_qtd_escolas_avaliacao",
    "escola_participacao_municipio",
]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

dev = gold[gold.ano==2023].copy()
test = gold[gold.ano==2024].copy()

X_dev,y_dev = dev[FEATURES],dev[TARGET].astype(int)
X_test,y_test = test[FEATURES],test[TARGET].astype(int)

def preprocessor():
    return ColumnTransformer([
        ("num",Pipeline([
            ("imputer",SimpleImputer(strategy="median")),
            ("scaler",StandardScaler())
        ]),NUMERIC_FEATURES),
        ("cat",Pipeline([
            ("imputer",SimpleImputer(strategy="most_frequent")),
            ("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
        ]),CATEGORICAL_FEATURES)
    ],verbose_feature_names_out=False)

def pipe(model):
    return Pipeline([("preprocessor",preprocessor()),("model",model)])

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring = {
    "f1":"f1",
    "recall":"recall",
    "precision":"precision",
    "roc_auc":"roc_auc",
    "pr_auc":"average_precision",
    "balanced_accuracy":"balanced_accuracy",
    "accuracy":"accuracy",
}

models = {
    "Dummy_All_Risk":DummyClassifier(strategy="constant",constant=1),
    "LogisticRegression":LogisticRegression(max_iter=1500,random_state=42),
    "RandomForest":RandomForestClassifier(
        n_estimators=220,min_samples_leaf=3,random_state=42,n_jobs=1
    ),
    "HistGradientBoosting":HistGradientBoostingClassifier(
        max_iter=180,learning_rate=0.07,random_state=42
    )
}

rows=[]
for name,model in models.items():
    print("Avaliando",name)
    s=cross_validate(pipe(model),X_dev,y_dev,cv=cv,scoring=scoring,n_jobs=1)
    row={"modelo":name}
    for metric in scoring:
        row[f"{metric}_media"]=float(np.mean(s[f"test_{metric}"]))
        row[f"{metric}_std"]=float(np.std(s[f"test_{metric}"]))
    rows.append(row)

comparison=pd.DataFrame(rows).sort_values(
    ["pr_auc_media","balanced_accuracy_media","f1_media"],ascending=False
).reset_index(drop=True)

display(comparison.round(4))
comparison.to_csv(REPORTS/"model_comparison.csv",index=False)

dummy=comparison[comparison.modelo=="Dummy_All_Risk"].iloc[0]
winner=comparison[comparison.modelo!="Dummy_All_Risk"].iloc[0]["modelo"]

print("BASELINE_F1=",round(dummy.f1_media,4))
print("BASELINE_PR_AUC=",round(dummy.pr_auc_media,4))
print("BASELINE_BAL_ACC=",round(dummy.balanced_accuracy_media,4))
print("MODELO_CANDIDATO=",winner)


## 7. Tuning em 2023


In [ ]:
if winner=="HistGradientBoosting":
    estimator=pipe(HistGradientBoostingClassifier(random_state=42))
    params={
        "model__learning_rate":loguniform(0.03,0.15),
        "model__max_iter":randint(120,241),
        "model__max_leaf_nodes":[15,31,63],
        "model__min_samples_leaf":[10,20,40],
        "model__l2_regularization":loguniform(1e-4,2.0),
    }
elif winner=="RandomForest":
    estimator=pipe(RandomForestClassifier(random_state=42,n_jobs=1))
    params={
        "model__n_estimators":randint(160,321),
        "model__max_depth":[10,16,20,None],
        "model__min_samples_leaf":[2,4,6,10],
        "model__min_samples_split":[2,8,16,24],
        "model__max_features":["sqrt","log2",0.5],
        "model__class_weight":[None,"balanced","balanced_subsample"],
        "model__max_samples":[0.75,0.9,None],
    }
else:
    estimator=pipe(LogisticRegression(max_iter=2000,random_state=42))
    params={
        "model__C":loguniform(1e-2,1e2),
        "model__class_weight":[None,"balanced"]
    }

cv_tune=StratifiedKFold(n_splits=3,shuffle=True,random_state=42)

search=RandomizedSearchCV(
    estimator,params,n_iter=10,scoring="average_precision",
    cv=cv_tune,random_state=42,n_jobs=-1,refit=True,verbose=1
)
search.fit(X_dev,y_dev)

print("BEST_CV_PR_AUC=",round(search.best_score_,4))
print("BEST_PARAMS=")
print(json.dumps(search.best_params_,indent=2,default=str))


## 8. Threshold OOF de 2023 e teste final 2024


In [ ]:
best=search.best_estimator_

oof= cross_val_predict(
    best,X_dev,y_dev,cv=cv_tune,method="predict_proba",n_jobs=-1
)[:,1]

thr=[]
for t in np.arange(0.20,0.801,0.01):
    yp=(oof>=t).astype(int)
    thr.append({
        "threshold":round(float(t),2),
        "f1":f1_score(y_dev,yp),
        "recall":recall_score(y_dev,yp),
        "precision":precision_score(y_dev,yp,zero_division=0),
        "balanced_accuracy":balanced_accuracy_score(y_dev,yp)
    })

thr=pd.DataFrame(thr).sort_values(
    ["f1","balanced_accuracy"],ascending=False
).reset_index(drop=True)

decision_threshold=float(thr.loc[0,"threshold"])
display(thr.head(12).round(4))
print("THRESHOLD=",decision_threshold)

prob=best.predict_proba(X_test)[:,1]
pred=(prob>=decision_threshold).astype(int)

metrics={
    "accuracy":accuracy_score(y_test,pred),
    "balanced_accuracy":balanced_accuracy_score(y_test,pred),
    "precision":precision_score(y_test,pred,zero_division=0),
    "recall":recall_score(y_test,pred),
    "f1":f1_score(y_test,pred),
    "roc_auc":roc_auc_score(y_test,prob),
    "pr_auc":average_precision_score(y_test,prob),
    "prevalence":float(y_test.mean()),
    "threshold":decision_threshold
}

baseline_pred=np.ones(len(y_test),dtype=int)
baseline_prob=np.ones(len(y_test),dtype=float)
baseline={
    "accuracy":accuracy_score(y_test,baseline_pred),
    "balanced_accuracy":balanced_accuracy_score(y_test,baseline_pred),
    "precision":precision_score(y_test,baseline_pred),
    "recall":recall_score(y_test,baseline_pred),
    "f1":f1_score(y_test,baseline_pred),
    "roc_auc":roc_auc_score(y_test,baseline_prob),
    "pr_auc":average_precision_score(y_test,baseline_prob)
}

display(pd.DataFrame([
    {"modelo":"Dummy_All_Risk",**baseline},
    {"modelo":winner,**{k:metrics[k] for k in baseline}}
]).round(4))

cm=confusion_matrix(y_test,pred)
print("CONFUSION_MATRIX=")
print(cm)
print("DELTA_F1=",round(metrics["f1"]-baseline["f1"],4))
print("DELTA_BAL_ACC=",round(metrics["balanced_accuracy"]-baseline["balanced_accuracy"],4))
print("DELTA_PR_AUC=",round(metrics["pr_auc"]-baseline["pr_auc"],4))

pd.DataFrame([metrics]).to_csv(REPORTS/"final_metrics_2024.csv",index=False)
pd.DataFrame([baseline]).to_csv(REPORTS/"baseline_2024.csv",index=False)


In [ ]:
disp=ConfusionMatrixDisplay(cm,display_labels=["Alfabetizado","Não alfabetizado"])
fig,ax=plt.subplots(figsize=(6,5))
disp.plot(ax=ax,values_format="d",colorbar=False)
ax.set_title("Matriz de confusão — teste temporal de 2024",pad=16)
fig.tight_layout()
fig.savefig(IMAGES/"confusion_matrix_2024.png",dpi=160,bbox_inches="tight")
plt.show()

fig,ax=plt.subplots(figsize=(6.5,5))
RocCurveDisplay.from_predictions(y_test,prob,ax=ax,name=winner)
ax.plot([0,1],[0,1],linestyle="--")
ax.set_title("Curva ROC — teste temporal de 2024")
fig.tight_layout()
fig.savefig(IMAGES/"roc_curve_2024.png",dpi=160,bbox_inches="tight")
plt.show()

fig,ax=plt.subplots(figsize=(6.5,5))
PrecisionRecallDisplay.from_predictions(y_test,prob,ax=ax,name=winner)
ax.axhline(y_test.mean(),linestyle="--",label="Baseline prevalência")
ax.legend()
ax.set_title("Curva Precision-Recall — teste temporal de 2024")
fig.tight_layout()
fig.savefig(IMAGES/"precision_recall_2024.png",dpi=160,bbox_inches="tight")
plt.show()


## 9. SHAP


In [ ]:
pre=best.named_steps["preprocessor"]
model=best.named_steps["model"]
names=np.array(pre.get_feature_names_out())

sample=X_test.sample(n=min(1500,len(X_test)),random_state=42)
Xt=pre.transform(sample)

if isinstance(model,LogisticRegression):
    expl=shap.LinearExplainer(model,Xt)
    vals=expl(Xt).values
else:
    expl=shap.TreeExplainer(model)
    raw=expl.shap_values(Xt)
    if isinstance(raw,list):
        vals=np.asarray(raw[1])
    else:
        arr=np.asarray(raw)
        vals=arr[:,:,1] if arr.ndim==3 else arr

sh=pd.DataFrame({
    "feature":names,
    "mean_abs_shap":np.abs(vals).mean(axis=0)
}).sort_values("mean_abs_shap",ascending=False)

def base_name(x):
    for c in CATEGORICAL_FEATURES:
        if x.startswith(c+"_"):
            return c
    return x

sh["variavel_base"]=sh.feature.map(base_name)
sh_global=(
    sh.groupby("variavel_base",as_index=False)["mean_abs_shap"]
      .sum().sort_values("mean_abs_shap",ascending=False)
)

display(sh_global.head(15).round(5))
sh_global.to_csv(REPORTS/"shap_global.csv",index=False)

plot=sh_global.head(15).sort_values("mean_abs_shap")
fig,ax=plt.subplots(figsize=(8,6))
ax.barh(plot.variavel_base,plot.mean_abs_shap)
ax.set_xlabel("Média de abs(SHAP)")
ax.set_title("Influência preditiva global")
fig.tight_layout()
fig.savefig(IMAGES/"shap_global.png",dpi=160,bbox_inches="tight")
plt.show()


## 10. Metas 2030: consumo da Gold da Fase 2 e análise prospectiva

A Fase 2 já tratava as metas municipais. Na tabela oficial utilizada, a meta de 2030
é auditada diretamente. Se `MIN = MAX`, o valor é usado como referência oficial comum,
sem depender de um join por ano que não corresponde à semântica da tabela.

A análise prospectiva compara o ritmo observado 2023→2024 com o ritmo necessário até 2030.
Com dois pontos anuais, isso é uma **projeção por cenário**, não um modelo formal de séries temporais.


### 10.1 Materialização e consumo da Gold territorial da Fase 2

Para tornar a continuidade técnica verificável, esta etapa clona o repositório da
Fase 2 e utiliza diretamente a função `add_meta_reference` de
`src/processing/gold_transform.py`.

A saída é materializada como `phase2_gold_territorial.parquet` e as células
prospectivas seguintes leem esse Parquet, em vez de reconstruir a camada territorial
no momento da análise.


In [ ]:
import subprocess
import importlib.util

PHASE2_LOCAL = Path("/content/techchallenge-fase2-pipeline-alfabetizacao")
PHASE2_GOLD_PATH = DATA / "phase2_gold_territorial.parquet"

if not PHASE2_LOCAL.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/acorrea79/techchallenge-fase2-pipeline-alfabetizacao.git",
        str(PHASE2_LOCAL)
    ], check=True)

gold_transform_path = PHASE2_LOCAL / "src" / "processing" / "gold_transform.py"
spec = importlib.util.spec_from_file_location("phase2_gold_transform", gold_transform_path)
phase2_gold_transform = importlib.util.module_from_spec(spec)
spec.loader.exec_module(phase2_gold_transform)

phase2_meta_sql = """
SELECT *
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio`
WHERE CAST(ano AS INT64) IN (2023, 2024)
"""
phase2_gold = client.query(phase2_meta_sql).to_dataframe(create_bqstorage_client=False)
phase2_gold["id_municipio"] = phase2_gold["id_municipio"].astype("string").str.zfill(7)

# A Gold da Fase 2 usa rótulos textuais de rede (ex.: "Municipal"),
# enquanto a base individual usa códigos INEP ("1".."4").
# Mantemos o valor original e criamos uma chave canônica para integração.
phase2_gold["rede_fase2"] = phase2_gold["rede"].astype("string").str.strip()

REDE_TO_CODE = {
    "federal": "1",
    "estadual": "2",
    "municipal": "3",
    "privada": "4",
    "1": "1",
    "2": "2",
    "3": "3",
    "4": "4",
}

phase2_gold["rede"] = (
    phase2_gold["rede_fase2"]
    .str.lower()
    .map(REDE_TO_CODE)
    .fillna(phase2_gold["rede_fase2"])
    .astype("string")
)

phase2_gold["ano"] = pd.to_numeric(phase2_gold["ano"], errors="coerce").astype("Int64")

# Usa diretamente a regra versionada na Fase 2.
phase2_gold = phase2_gold_transform.add_meta_reference(phase2_gold)
phase2_gold["gold_dataset"] = "phase2_gold_territorial_consumed_by_phase3"
phase2_gold["phase2_repository"] = PHASE2_REPO
phase2_gold.to_parquet(PHASE2_GOLD_PATH, index=False)

print("PHASE2_GOLD_PATH=", PHASE2_GOLD_PATH)
print("PHASE2_GOLD_SHAPE=", phase2_gold.shape)
print("PHASE2_GOLD_2030_COVERAGE=", round(
    100 * phase2_gold["meta_alfabetizacao_2030"].notna().mean(), 2
), "%")

display(phase2_gold[[
    "ano", "id_municipio", "rede", "taxa_alfabetizacao",
    "meta_alfabetizacao_2030", "distancia_meta_2030_pontos", "status_meta"
]].head())


In [ ]:
phase2_gold = pd.read_parquet(PHASE2_GOLD_PATH)
meta_non_null = pd.to_numeric(
    phase2_gold["meta_alfabetizacao_2030"], errors="coerce"
).dropna()

meta_audit = pd.DataFrame([{
    "meta_2030_min": meta_non_null.min(),
    "meta_2030_max": meta_non_null.max(),
    "registros_com_meta": int(meta_non_null.shape[0]),
}])
display(meta_audit)

meta_min=float(meta_audit.loc[0,"meta_2030_min"])
meta_max=float(meta_audit.loc[0,"meta_2030_max"])
if abs(meta_min-meta_max)>1e-9:
    raise RuntimeError("A meta 2030 não é constante na Gold da Fase 2.")
META_2030=meta_max
print("META_2030_OFICIAL=",META_2030)


In [ ]:
# Trajetórias 2023→2024 consumidas da Gold territorial da Fase 2.
phase2_gold = pd.read_parquet(PHASE2_GOLD_PATH)
phase2_gold["taxa_alfabetizacao"] = pd.to_numeric(
    phase2_gold["taxa_alfabetizacao"], errors="coerce"
)

rates = phase2_gold[[
    "ano", "id_municipio", "rede", "taxa_alfabetizacao",
    "meta_alfabetizacao_2030"
]].dropna(subset=["ano","id_municipio","rede","taxa_alfabetizacao"]).copy()

rates["ano"] = pd.to_numeric(rates["ano"], errors="coerce").astype("Int64")
rates = rates[rates["ano"].isin([2023, 2024])]

wide = rates.pivot_table(
    index=["id_municipio","rede"],
    columns="ano",
    values="taxa_alfabetizacao",
    aggfunc="mean"
).reset_index()
wide.columns.name=None
wide=wide.rename(columns={2023:"taxa_2023",2024:"taxa_2024"})
wide=wide.dropna(subset=["taxa_2023","taxa_2024"]).copy()

wide["meta_alfabetizacao_2030"] = META_2030
wide["ritmo_observado_pp_ano"] = wide["taxa_2024"] - wide["taxa_2023"]
wide["ritmo_necessario_pp_ano"] = np.maximum(
    0,(META_2030-wide["taxa_2024"])/6.0
)
wide["cenario_tendencial_2030"] = (
    wide["taxa_2024"] + 6*wide["ritmo_observado_pp_ano"]
).clip(0,100)
wide["gap_tendencial_meta_2030"] = wide["cenario_tendencial_2030"] - META_2030
wide["risco_nao_atingir_meta_2030"] = wide["gap_tendencial_meta_2030"] < 0
wide["ritmo_insuficiente"] = wide["ritmo_observado_pp_ano"] < wide["ritmo_necessario_pp_ano"]

print("REGISTROS_COM_TRAJETORIA=",len(wide))
print("RISCO_PROSPECTIVO_2030=",int(wide.risco_nao_atingir_meta_2030.sum()))
wide.to_csv(REPORTS/"prospective_2030.csv",index=False)


In [ ]:
pred_df=test[
    ["id_municipio","rede","sigla_uf","regiao","id_escola",TARGET]
].copy()
pred_df["prob_nao_alfabetizado"]=prob

risk=(
    pred_df.groupby(["id_municipio","rede","sigla_uf","regiao"],as_index=False)
      .agg(
        alunos_amostra=("id_escola","size"),
        escolas_amostra=("id_escola","nunique"),
        score_risco_modelo=("prob_nao_alfabetizado","mean")
      )
)

# Normalização defensiva das chaves antes do merge.
risk["id_municipio"] = risk["id_municipio"].astype("string").str.zfill(7)
risk["rede"] = risk["rede"].astype("string").str.strip()

wide["id_municipio"] = wide["id_municipio"].astype("string").str.zfill(7)
wide["rede"] = wide["rede"].astype("string").str.strip()

policy=wide.merge(
    risk,
    on=["id_municipio","rede"],
    how="left",
    validate="one_to_one"
)

merge_match_rate = 100 * policy["score_risco_modelo"].notna().mean()
print("POLICY_MODEL_SCORE_COVERAGE_PCT=", round(merge_match_rate, 2))

priority=policy[
    policy.risco_nao_atingir_meta_2030
    & policy["score_risco_modelo"].notna()
].copy()

priority=priority.sort_values(
    ["score_risco_modelo","gap_tendencial_meta_2030"],
    ascending=[False,True]
).reset_index(drop=True)

priority["ranking_prioridade_2030"]=priority.index+1

print("RISCO_PROSPECTIVO_TOTAL=", int(policy.risco_nao_atingir_meta_2030.sum()))
print("RISCO_PRIORIZAVEL_COM_SCORE=", len(priority))

if len(priority) == 0:
    raise RuntimeError(
        "Ranking prospectivo sem scores do modelo. Verifique a normalização das chaves."
    )

display(priority.head(20).round(3))
priority.to_csv(REPORTS/"ranking_prospectivo_2030.csv",index=False)


## 11. Resumo final


In [ ]:
summary={
    "phase2_repo":PHASE2_REPO,
    "school_id_match_rate_pct":float(school_id_match_rate),
    "gold_shape":list(gold.shape),
    "gold_sha256":gold_sha256,
    "school_feature_coverage_pct":float(school_feature_coverage),
    "municipality_network_with_school_variation_pct":float(school_variation_pct),
    "features":FEATURES,
    "winner":winner,
    "best_cv_pr_auc":float(search.best_score_),
    "threshold":float(decision_threshold),
    "baseline_2024":{k:float(v) for k,v in baseline.items()},
    "metrics_2024":{k:float(v) for k,v in metrics.items()},
    "delta_f1":float(metrics["f1"]-baseline["f1"]),
    "delta_balanced_accuracy":float(
        metrics["balanced_accuracy"]-baseline["balanced_accuracy"]
    ),
    "delta_pr_auc":float(metrics["pr_auc"]-baseline["pr_auc"]),
    "meta_2030":float(META_2030),
    "prospective_rows":int(len(wide)),
    "prospective_risk_2030":int(wide.risco_nao_atingir_meta_2030.sum()),
    "policy_model_score_coverage_pct":float(merge_match_rate),
    "prospective_prioritizable_with_score":int(len(priority)),
    "top_shap":sh_global.head(10).to_dict(orient="records")
}

(REPORTS/"results_summary.json").write_text(
    json.dumps(summary,ensure_ascii=False,indent=2),
    encoding="utf-8"
)

joblib.dump(best,MODELS/"final_model.joblib")

status=(
    school_feature_coverage>=99
    and school_variation_pct>=50
    and metrics["balanced_accuracy"]>baseline["balanced_accuracy"]
    and metrics["pr_auc"]>baseline["pr_auc"]
    and len(wide)>0
    and wide.risco_nao_atingir_meta_2030.sum()>0
)

print("VALIDACAO_METODOLOGICA=", "APROVADA" if status else "REVISAR")
print()
print("=== RESUMO DA EXECUCAO ===")
print("SCHOOL_ID_MATCH_RATE_PCT=",round(school_id_match_rate,4))
print("GOLD_SHAPE=",gold.shape)
print("SCHOOL_FEATURE_COVERAGE_PCT=",round(school_feature_coverage,2))
print("SCHOOL_VARIATION_PCT=",round(school_variation_pct,2))
print("FEATURES=",FEATURES)
print("MODELO_CANDIDATO=",winner)
print("BASELINE_2024=",json.dumps(baseline))
print("METRICS_2024=",json.dumps(metrics))
print("DELTA_F1=",round(metrics["f1"]-baseline["f1"],4))
print("DELTA_BAL_ACC=",round(metrics["balanced_accuracy"]-baseline["balanced_accuracy"],4))
print("DELTA_PR_AUC=",round(metrics["pr_auc"]-baseline["pr_auc"],4))
print("META_2030_OFICIAL=",META_2030)
print("RISCO_PROSPECTIVO_2030=",int(wide.risco_nao_atingir_meta_2030.sum()))
print("POLICY_MODEL_SCORE_COVERAGE_PCT=",round(merge_match_rate,2))
print("RISCO_PRIORIZAVEL_COM_SCORE=",len(priority))
print("TOP_SHAP=")
display(sh_global.head(10).round(5))
print("TOP20_PROSPECTIVO=")
display(priority.head(20).round(3))
print("VALIDACAO_METODOLOGICA=", "APROVADA" if status else "REVISAR")


In [ ]:
archive=shutil.make_archive(
    "/content/tech_challenge_fase3_v3_artifacts",
    "zip",
    root_dir=OUT
)
print("ARQUIVO_DE_ARTEFATOS=",archive)
